In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

usage = spark.table("slv_usage_events")
subscriptions = spark.table("slv_subscriptions")
plans = spark.table("slv_plans")
customers = spark.table("slv_customers")
customer_mrr = spark.table("gld_customer_monthly_mrr")

print(f"Usage rows:         {usage.count():,}")
print(f"Subscription rows:  {subscriptions.count():,}")
print(f"Customer MRR rows:  {customer_mrr.count():,}")

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 3, Finished, Available, Finished, False)

Usage rows:         349,318
Subscription rows:  754
Customer MRR rows:  12,855


In [3]:
#Aggregate usage by customer and month

monthly_usage = (
    usage
    .withColumn(
        "MonthStart",
        F.trunc(F.col("EventDate"), "month")
    )
    .groupBy(
        "CustomerID",
        "SubscriptionID",
        "MonthStart"
    )
    .agg(
        F.sum("WorkflowsRun").alias("WorkflowsRun"),
        F.sum("APICalls").alias("APICalls"),
        F.sum("CreditsConsumed").alias("CreditsConsumed"),
        F.max("ActiveUsers").alias("PeakActiveUsers"),
        F.avg("ActiveUsers").alias("AverageActiveUsers"),
        F.countDistinct("EventDate").alias("ActiveDays")
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 5, Finished, Available, Finished, False)

In [4]:
#Add subscription and plan information

monthly_usage_enriched = (
    monthly_usage.alias("u")
    .join(
        subscriptions.alias("s"),
        F.col("u.SubscriptionID") == F.col("s.SubscriptionID"),
        "left"
    )
    .join(
        plans.alias("p"),
        F.col("s.PlanID") == F.col("p.PlanID"),
        "left"
    )
    .select(
        F.col("u.CustomerID"),
        F.col("u.SubscriptionID"),
        F.col("u.MonthStart"),
        F.col("s.PlanID"),
        F.col("p.PlanName"),
        F.col("p.IncludedCredits"),
        F.col("u.WorkflowsRun"),
        F.col("u.APICalls"),
        F.col("u.CreditsConsumed"),
        F.col("u.PeakActiveUsers"),
        F.round(F.col("u.AverageActiveUsers"), 2).alias("AverageActiveUsers"),
        F.col("u.ActiveDays")
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 6, Finished, Available, Finished, False)

In [5]:
#Aggregate to customer-month grain

gld_customer_monthly_usage = (
    monthly_usage_enriched
    .groupBy("CustomerID", "MonthStart")
    .agg(
        F.sum("WorkflowsRun").alias("WorkflowsRun"),
        F.sum("APICalls").alias("APICalls"),
        F.sum("CreditsConsumed").alias("CreditsConsumed"),
        F.sum("IncludedCredits").alias("IncludedCredits"),
        F.max("PeakActiveUsers").alias("PeakActiveUsers"),
        F.round(
            F.avg("AverageActiveUsers"),
            2
        ).alias("AverageActiveUsers"),
        F.max("ActiveDays").alias("ActiveDays")
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 7, Finished, Available, Finished, False)

In [6]:
#Calculate credit utilisation

gld_customer_monthly_usage = (
    gld_customer_monthly_usage
    .withColumn(
        "CreditUtilisationRate",
        F.when(
            F.col("IncludedCredits") > 0,
            F.col("CreditsConsumed") / F.col("IncludedCredits")
        ).otherwise(F.lit(None))
        .cast("decimal(10,4)")
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 8, Finished, Available, Finished, False)

In [7]:
#Add previous-month usage

usage_window = (
    Window
    .partitionBy("CustomerID")
    .orderBy("MonthStart")
)

gld_customer_monthly_usage = (
    gld_customer_monthly_usage
    .withColumn(
        "PreviousMonthCredits",
        F.lag("CreditsConsumed").over(usage_window)
    )
    .withColumn(
        "PreviousMonthWorkflows",
        F.lag("WorkflowsRun").over(usage_window)
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 9, Finished, Available, Finished, False)

In [8]:
#Calculate usage movement

gld_customer_monthly_usage = (
    gld_customer_monthly_usage
    .withColumn(
        "CreditUsageChange",
        F.col("CreditsConsumed")
        - F.coalesce(F.col("PreviousMonthCredits"), F.lit(0))
    )
    .withColumn(
        "CreditUsageChangePct",
        F.when(
            F.col("PreviousMonthCredits") > 0,
            (
                F.col("CreditsConsumed")
                - F.col("PreviousMonthCredits")
            ) / F.col("PreviousMonthCredits")
        ).otherwise(F.lit(None))
        .cast("decimal(10,4)")
    )
    .withColumn(
        "WorkflowChangePct",
        F.when(
            F.col("PreviousMonthWorkflows") > 0,
            (
                F.col("WorkflowsRun")
                - F.col("PreviousMonthWorkflows")
            ) / F.col("PreviousMonthWorkflows")
        ).otherwise(F.lit(None))
        .cast("decimal(10,4)")
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 10, Finished, Available, Finished, False)

In [9]:
#Add customer information

gld_customer_monthly_usage = (
    gld_customer_monthly_usage.alias("u")
    .join(
        customers.alias("c"),
        F.col("u.CustomerID") == F.col("c.CustomerID"),
        "left"
    )
    .select(
        F.col("u.CustomerID"),
        F.col("c.CustomerName"),
        F.col("c.Industry"),
        F.col("c.Country"),
        F.col("c.CompanySize"),
        F.col("u.MonthStart"),
        F.col("u.WorkflowsRun"),
        F.col("u.APICalls"),
        F.col("u.CreditsConsumed"),
        F.col("u.IncludedCredits"),
        F.col("u.CreditUtilisationRate"),
        F.col("u.PeakActiveUsers"),
        F.col("u.AverageActiveUsers"),
        F.col("u.ActiveDays"),
        F.col("u.PreviousMonthCredits"),
        F.col("u.CreditUsageChange"),
        F.col("u.CreditUsageChangePct"),
        F.col("u.WorkflowChangePct")
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 11, Finished, Available, Finished, False)

In [10]:
display(
    gld_customer_monthly_usage
    .filter(F.col("CreditUsageChangePct").isNotNull())
    .orderBy(F.col("CreditUsageChangePct").asc())
    .limit(5
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d54b2804-9b16-4b30-8d1e-f157d76786a1)

In [11]:
#Combine MRR and usage

customer_health_base = (
    customer_mrr.alias("m")
    .join(
        gld_customer_monthly_usage.alias("u"),
        (
            (F.col("m.CustomerID") == F.col("u.CustomerID"))
            & (F.col("m.MonthStart") == F.col("u.MonthStart"))
        ),
        "left"
    )
    .select(
        F.col("m.CustomerID"),
        F.col("m.CustomerName"),
        F.col("m.Industry"),
        F.col("m.Country"),
        F.col("m.CompanySize"),
        F.col("m.AcquisitionChannel"),
        F.col("m.MonthStart"),
        F.col("m.OpeningMRR"),
        F.col("m.ClosingMRR"),
        F.col("m.NewMRR"),
        F.col("m.ExpansionMRR"),
        F.col("m.ContractionMRR"),
        F.col("m.ChurnedMRR"),
        F.col("u.WorkflowsRun"),
        F.col("u.APICalls"),
        F.col("u.CreditsConsumed"),
        F.col("u.CreditUtilisationRate"),
        F.col("u.AverageActiveUsers"),
        F.col("u.ActiveDays"),
        F.col("u.CreditUsageChangePct")
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 13, Finished, Available, Finished, False)

In [12]:
#Create a simple health classification

gld_customer_health = (
    customer_health_base

    .withColumn(
        "NoUsageFlag",
        F.when(
            (F.col("ClosingMRR") > 0)
            & (
                F.col("CreditsConsumed").isNull()
                | (F.col("CreditsConsumed") == 0)
            ),
            1
        ).otherwise(0)
    )

    .withColumn(
        "UsageDeclineFlag",
        F.when(
            F.col("CreditUsageChangePct") <= -0.30,
            1
        ).otherwise(0)
    )

    .withColumn(
        "LowActiveDaysFlag",
        F.when(
            (F.col("ClosingMRR") > 0)
            & (F.coalesce(F.col("ActiveDays"), F.lit(0)) < 5),
            1
        ).otherwise(0)
    )

    .withColumn(
        "ContractionFlag",
        F.when(F.col("ContractionMRR") > 0, 1).otherwise(0)
    )

    .withColumn(
        "RiskScore",
        (
            F.col("NoUsageFlag") * 3
            + F.col("UsageDeclineFlag") * 2
            + F.col("LowActiveDaysFlag")
            + F.col("ContractionFlag") * 2
        )
    )

    .withColumn(
        "CustomerHealth",
        F.when(
            F.col("ChurnedMRR") > 0,
            "Churned"
        )
        .when(
            F.col("RiskScore") >= 4,
            "At Risk"
        )
        .when(
            F.col("RiskScore") >= 2,
            "Watch"
        )
        .otherwise("Healthy")
    )
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 14, Finished, Available, Finished, False)

In [13]:
#Inspect the health distribution

display(
    gld_customer_health
    .groupBy("MonthStart", "CustomerHealth")
    .agg(
        F.countDistinct("CustomerID").alias("Customers"),
        F.sum("ClosingMRR").alias("MRR")
    )
    .orderBy("MonthStart", "CustomerHealth")
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df3bd6f9-e949-45a6-8721-63e7c0a5cfbc)

In [15]:
#inspect current at-risk customers:

display(
    gld_customer_health
    .filter(
        (F.col("MonthStart") == F.to_date(F.lit("2026-06-01")))
        & (F.col("CustomerHealth") == "At Risk")
    )
    .orderBy(F.col("RiskScore").desc())
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5538807f-4e22-43f8-baa7-e73ff1e292d7)

In [16]:
#Validate uniqueness

usage_duplicates = (
    gld_customer_monthly_usage
    .groupBy("CustomerID", "MonthStart")
    .count()
    .filter(F.col("count") > 1)
)

health_duplicates = (
    gld_customer_health
    .groupBy("CustomerID", "MonthStart")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate customer-month usage rows:",
    usage_duplicates.count()
)

print(
    "Duplicate customer-month health rows:",
    health_duplicates.count()
)

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 18, Finished, Available, Finished, False)

Duplicate customer-month usage rows: 0
Duplicate customer-month health rows: 0


In [17]:
#Write the Gold tables

(
    gld_customer_monthly_usage.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gld_customer_monthly_usage")
)

(
    gld_customer_health.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gld_customer_health")
)

print("Gold product-usage and customer-health tables created successfully.")

StatementMeta(, d2e113d5-d934-4bb3-b88c-c9bdf6bb7792, 19, Finished, Available, Finished, False)

Gold product-usage and customer-health tables created successfully.
